In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
"""
HuBERT Sequence Embedding Extraction for Speech Emotion Recognition
------------------------------------------------------------------
Preprocessing:
- Load WAV audio
- Convert to mono
- Resample to 16kHz
- Trim silence
- Normalize amplitude

Feature Extraction:
- HuBERT hidden states
- Keep temporal sequence embeddings (NO mean pooling)

Output:
- Saves [T x 768] sequence embeddings as .npy files
"""

# !pip install torch transformers librosa soundfile tqdm numpy pandas

import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm
import torch
from transformers import HubertModel, Wav2Vec2FeatureExtractor

# =========================================================
# CONFIG
# =========================================================

AUDIO_DIR = "/content/drive/MyDrive/IIITH_Voice/TESS_Toronto_emotional_speech_set_data"
OUTPUT_DIR = "/content/drive/MyDrive/IIITH_Voice/TESS_Time_Embeddings_hubert"

MODEL_NAME = "facebook/hubert-base-ls960"

SAMPLE_RATE = 16000

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# LOAD DATASET
# =========================================================

records = []

for root, _, files in os.walk(AUDIO_DIR):

    for f in files:

        if f.lower().endswith(".wav"):

            label = os.path.basename(root)

            path = os.path.join(root, f)

            records.append({
                "path": path,
                "label": label
            })

df = pd.DataFrame(records)

print(f"\nTotal audio files found: {len(df)}")

print("\nOriginal label distribution:\n")
print(df["label"].value_counts())

# =========================================================
# BALANCE DATASET
# =========================================================

min_count = df["label"].value_counts().min()

df_balanced = (
    df.groupby("label", group_keys=False)
      .sample(n=min_count, random_state=42)
)

print("\nBalanced dataset summary:")
print(f"→ Using {len(df_balanced)} total files ({min_count} per class)\n")

print(df_balanced["label"].value_counts())

# =========================================================
# LOAD HUBERT
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)

hubert = HubertModel.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True
).to(device)

hubert.eval()

# =========================================================
# PREPROCESS + FEATURE EXTRACTION
# =========================================================

def extract_hubert_sequence(path, sr=SAMPLE_RATE):

    # -----------------------------------------------------
    # LOAD AUDIO
    # -----------------------------------------------------

    waveform, sr = librosa.load(
        path,
        sr=sr,
        mono=True
    )

    # -----------------------------------------------------
    # TRIM SILENCE
    # -----------------------------------------------------

    waveform, _ = librosa.effects.trim(
        waveform,
        top_db=20
    )

    # -----------------------------------------------------
    # NORMALIZE AMPLITUDE
    # -----------------------------------------------------

    waveform = librosa.util.normalize(waveform)

    # -----------------------------------------------------
    # FEATURE EXTRACTOR
    # -----------------------------------------------------

    inputs = feature_extractor(
        waveform,
        sampling_rate=sr,
        return_tensors="pt",
        padding=True
    )

    input_values = inputs["input_values"].to(device)

        # -----------------------------------------------------
    # HUBERT FORWARD PASS
    # -----------------------------------------------------

    with torch.no_grad():

        outputs = hubert(input_values)

        # all 13 hidden layers
        hidden_states = outputs.hidden_states

    # -----------------------------------------------------
    # USE MIDDLE + UPPER LAYERS
    # layers 6–12
    # -----------------------------------------------------

    selected_layers = hidden_states[6:13]

    # -----------------------------------------------------
    # STACK LAYERS
    # shape:
    # [7, 1, T, 768]
    # -----------------------------------------------------

    stacked_layers = torch.stack(
        selected_layers,
        dim=0
    )

    # -----------------------------------------------------
    # EQUAL-WEIGHT LAYER AVERAGE
    # shape:
    # [1, T, 768]
    # -----------------------------------------------------

    weighted_sum = stacked_layers.mean(dim=0)

    # -----------------------------------------------------
    # REMOVE BATCH DIMENSION
    # [1, T, 768] → [T, 768]
    # -----------------------------------------------------

    sequence_embedding = (
        weighted_sum
        .squeeze(0)
        .cpu()
        .numpy()
    )

    return sequence_embedding

# =========================================================
# EXTRACT + SAVE EMBEDDINGS
# =========================================================

print("\nExtracting HuBERT sequence embeddings...\n")

for _, row in tqdm(
    df_balanced.iterrows(),
    total=len(df_balanced),
    desc="Processing audio files"
):

    sequence_embedding = extract_hubert_sequence(row["path"])

    label_folder = os.path.join(
        OUTPUT_DIR,
        row["label"]
    )

    os.makedirs(label_folder, exist_ok=True)

    base = os.path.splitext(
        os.path.basename(row["path"])
    )[0]

    save_path = os.path.join(
        label_folder,
        f"{base}.npy"
    )

    np.save(save_path, sequence_embedding)

print(
    f"\n✅ Done! Saved sequence embeddings to:\n{OUTPUT_DIR}"
)

# =========================================================
# OPTIONAL DEBUG
# =========================================================

sample = np.load(save_path)

print("\nExample embedding shape:")

print(sample.shape)

# Example:
# (249, 768)
#
# 249 = temporal frames
# 768 = HuBERT embedding dimension


Total audio files found: 5600

Original label distribution:

label
YAF_disgust               400
YAF_sad                   400
YAF_happy                 400
YAF_fear                  400
YAF_pleasant_surprised    400
YAF_neutral               400
OAF_Sad                   400
YAF_angry                 400
OAF_Pleasant_surprise     400
OAF_neutral               400
OAF_Fear                  400
OAF_happy                 400
OAF_angry                 400
OAF_disgust               400
Name: count, dtype: int64

Balanced dataset summary:
→ Using 5600 total files (400 per class)

label
OAF_Fear                  400
OAF_Pleasant_surprise     400
OAF_Sad                   400
OAF_angry                 400
OAF_disgust               400
OAF_happy                 400
OAF_neutral               400
YAF_angry                 400
YAF_disgust               400
YAF_fear                  400
YAF_happy                 400
YAF_neutral               400
YAF_pleasant_surprised    400
YAF_sad              

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]


Extracting HuBERT sequence embeddings...



Streaming output truncated to the last 5000 lines.
Processing audio files: 100%|██████████| 5600/5600 [31:34<00:00,  2.96it/s]


✅ Done! Saved sequence embeddings to:
//content/drive/MyDrive/IIITH_Voice/TESS_Time_Embeddings_hubert

Example embedding shape:
(103, 768)


In [ ]:
!apt-get install tree -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  tree
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 47.9 kB of archives.
After this operation, 116 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 47.9 kB in 0s (140 kB/s)
Selecting previously unselected package tree.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../tree_2.0.2-1_amd64.deb ...
Unpacking tree (2.0.2-1) ...
Setting up tree (2.0.2-1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!tree -d /content/drive/MyDrive/IIITH_Voice

/content/drive/MyDrive/IIITH_Voice
├── TESS_Time_Embeddings_hubert
│   ├── OAF_angry
│   ├── OAF_disgust
│   ├── OAF_Fear
│   ├── OAF_happy
│   ├── OAF_neutral
│   ├── OAF_Pleasant_surprise
│   ├── OAF_Sad
│   ├── YAF_angry
│   ├── YAF_disgust
│   ├── YAF_fear
│   ├── YAF_happy
│   ├── YAF_neutral
│   ├── YAF_pleasant_surprised
│   └── YAF_sad
└── TESS_Toronto_emotional_speech_set_data
    ├── OAF_angry
    ├── OAF_disgust
    ├── OAF_Fear
    ├── OAF_happy
    ├── OAF_neutral
    ├── OAF_Pleasant_surprise
    ├── OAF_Sad
    ├── YAF_angry
    ├── YAF_disgust
    ├── YAF_fear
    ├── YAF_happy
    ├── YAF_neutral
    ├── YAF_pleasant_surprised
    └── YAF_sad

30 directories


In [ ]:
!rm -rf "/content/drive/MyDrive/IIITH_Voice/TESS_Toronto_emotional_speech_set_data/TESS Toronto emotional speech set data"

In [ ]:
import os

base_dir = "/content/drive/MyDrive/IIITH_Voice/TESS_Toronto_emotional_speech_set_data"

for folder in sorted(os.listdir(base_dir)):

    folder_path = os.path.join(base_dir, folder)

    if os.path.isdir(folder_path):

        n_files = len([
            f for f in os.listdir(folder_path)
            if os.path.isfile(os.path.join(folder_path, f))
        ])

        print(f"{folder:<30} {n_files}")

OAF_Fear                       200
OAF_Pleasant_surprise          200
OAF_Sad                        200
OAF_angry                      200
OAF_disgust                    200
OAF_happy                      200
OAF_neutral                    200
YAF_angry                      200
YAF_disgust                    200
YAF_fear                       200
YAF_happy                      200
YAF_neutral                    200
YAF_pleasant_surprised         200
YAF_sad                        200


In [ ]:
import os

base_dir = "/content/drive/MyDrive/IIITH_Voice/TESS_Time_Embeddings_hubert"

for folder in sorted(os.listdir(base_dir)):

    folder_path = os.path.join(base_dir, folder)

    if os.path.isdir(folder_path):

        n_files = len([
            f for f in os.listdir(folder_path)
            if os.path.isfile(os.path.join(folder_path, f))
        ])

        print(f"{folder:<30} {n_files}")

OAF_Fear                       200
OAF_Pleasant_surprise          200
OAF_Sad                        200
OAF_angry                      200
OAF_disgust                    200
OAF_happy                      200
OAF_neutral                    200
YAF_angry                      200
YAF_disgust                    200
YAF_fear                       200
YAF_happy                      200
YAF_neutral                    200
YAF_pleasant_surprised         200
YAF_sad                        200


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm

In [ ]:
EMBEDDING_DIR = "/content/drive/MyDrive/IIITH_Voice/TESS_Time_Embeddings_hubert"

BATCH_SIZE = 16
HIDDEN_SIZE = 128
NUM_EPOCHS = 20
LEARNING_RATE = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
paths = []
labels = []

for label in os.listdir(EMBEDDING_DIR):

    label_dir = os.path.join(EMBEDDING_DIR, label)

    if os.path.isdir(label_dir):

        # Convert to lowercase
        normalized_label = label.lower()

        # Remove speaker prefix (oaf_ / yaf_)
        normalized_label = normalized_label.replace("oaf_", "")
        normalized_label = normalized_label.replace("yaf_", "")

        # Fix inconsistent naming
        normalized_label = normalized_label.replace(
            "pleasant_surprised",
            "pleasant_surprise"
        )

        for file in os.listdir(label_dir):

            if file.endswith(".npy"):

                paths.append(
                    os.path.join(label_dir, file)
                )

                labels.append(normalized_label)

print(f"Total files: {len(paths)}")

Total files: 2800


In [ ]:
label_encoder = LabelEncoder()

encoded_labels = label_encoder.fit_transform(labels)

num_classes = len(label_encoder.classes_)

print(label_encoder.classes_)

['angry' 'disgust' 'fear' 'happy' 'neutral' 'pleasant_surprise' 'sad']


In [ ]:
train_paths, test_paths, train_labels, test_labels = train_test_split(
    paths,
    encoded_labels,
    test_size=0.2,
    random_state=42,
    stratify=encoded_labels
)

In [ ]:
class EmotionDataset(Dataset):

    def __init__(self, paths, labels):

        self.paths = paths
        self.labels = labels

    def __len__(self):

        return len(self.paths)

    def __getitem__(self, idx):

        embedding = np.load(self.paths[idx])

        embedding = torch.tensor(
            embedding,
            dtype=torch.float
        )

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return embedding, label

In [ ]:

def collate_fn(batch):

    embeddings = [item[0] for item in batch]

    labels = torch.tensor(
        [item[1] for item in batch]
    )

    lengths = torch.tensor(
        [emb.shape[0] for emb in embeddings]
    )

    padded_embeddings = pad_sequence(
        embeddings,
        batch_first=True
    )

    return padded_embeddings, labels, lengths

In [ ]:
train_dataset = EmotionDataset(
    train_paths,
    train_labels
)


test_dataset = EmotionDataset(
    test_paths,
    test_labels
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
class Attention(nn.Module):

    def __init__(self, hidden_size):

        super().__init__()

        self.attention = nn.Linear(
            hidden_size * 2,
            1
        )

    def forward(self, lstm_outputs, lengths):

        scores = self.attention(
            lstm_outputs
        ).squeeze(-1)

        max_len = lstm_outputs.size(1)

        mask = torch.arange(
            max_len,
            device=lengths.device
        )[None, :] < lengths[:, None]

        scores[~mask] = -1e9

        weights = torch.softmax(
            scores,
            dim=1
        )

        weights = weights.unsqueeze(-1)

        context = torch.sum(
            weights * lstm_outputs,
            dim=1
        )

        return context

In [ ]:
class EmotionModel(nn.Module):

    def __init__(self, hidden_size, num_classes):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.attention = Attention(hidden_size)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x, lengths):

        packed = nn.utils.rnn.pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_outputs, _ = self.lstm(packed)

        lstm_outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_outputs,
            batch_first=True
        )

        attention_output = self.attention(
            lstm_outputs,
            lengths
        )

        logits = self.classifier(
            attention_output
        )

        return logits

In [ ]:
print(num_classes)

7


In [ ]:
model = EmotionModel(
    hidden_size=HIDDEN_SIZE,
    num_classes=num_classes
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
     lr=LEARNING_RATE
)

In [ ]:
best_acc = 0

for epoch in range(NUM_EPOCHS):

    # ==========================================
    # TRAINING
    # ==========================================

    model.train()

    total_loss = 0

    for embeddings, labels, lengths in tqdm(train_loader):

        embeddings = embeddings.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(embeddings, lengths)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # ==========================================
    # VALIDATION
    # ==========================================

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for embeddings, labels, lengths in test_loader:

            embeddings = embeddings.to(DEVICE)

            outputs = model(embeddings, lengths)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )

    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"Loss: {avg_loss:.4f} | "
        f"Val Acc: {accuracy:.4f}"
    )

    # ==========================================
    # SAVE BEST MODEL
    # ==========================================

    if accuracy > best_acc:

        best_acc = accuracy

        torch.save(
            model.state_dict(),
            "/content/best_emotion_model.pth"
        )

        print("✅ Best model saved!")

  0%|          | 0/140 [00:01<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for embeddings, labels, lengths in test_loader:

        embeddings = embeddings.to(DEVICE)

        outputs = model(embeddings, lengths)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())

        all_labels.extend(labels.numpy())

In [ ]:
accuracy = accuracy_score(
    all_labels,
    all_preds
)

print(f"\nTest Accuracy: {accuracy:.4f}")


Test Accuracy: 1.0000


In [ ]:
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=label_encoder.classes_
    )
)

                   precision    recall  f1-score   support

            angry       1.00      1.00      1.00        80
          disgust       1.00      1.00      1.00        80
             fear       1.00      1.00      1.00        80
            happy       1.00      1.00      1.00        80
          neutral       1.00      1.00      1.00        80
pleasant_surprise       1.00      1.00      1.00        80
              sad       1.00      1.00      1.00        80

         accuracy                           1.00       560
        macro avg       1.00      1.00      1.00       560
     weighted avg       1.00      1.00      1.00       560

